# GEM Pipeline & Terminal Data Export

Exports LNG terminals and pipelines from Google Sheets to Excel/GeoJSON/GeoPackage.

**Pipeline modes:** Oil-NGL | Oil | NGL | Gas | Gas-Hydrogen | Hydrogen | Oil-and-Gas (combined)

**Optional:** Filter by country list

In [ ]:
%pip install -q -e ../../gem-tracker-constants

import os
import re
import datetime
import shutil
import time
import zipfile
from pathlib import Path

import pandas as pd
import geopandas as gpd
import pygsheets
import shapely.geometry
from shapely import from_wkt, set_precision

# Canonical fuel buckets. Source of truth: the in-repo gem-tracker-constants
# package (repo root, installed editable above).
# OIL_FUEL_OPTIONS / NGL_FUEL_OPTIONS define which raw Fuel values qualify as
# Oil / NGL pipelines — used by simplify_fuel_types() for the simplified
# release downloads, and by the QC summary sheets, so the two align.
from gem_tracker_constants import (
    GAS_FUEL_OPTIONS,
    GAS_HYDROGEN_FUEL_OPTIONS,
    HYDROGEN_FUEL_OPTIONS,
    OIL_NGL_COMBINED,
    OIL_FUEL_OPTIONS,
    NGL_FUEL_OPTIONS,
    collapse_gas_and_hydrogen,
)

print("✓ Imports loaded")

## Configuration

In [ ]:
# Sheets
#PIPELINES_SHEET_KEY = '1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek'
PIPELINES_SHEET_KEY = '1ozthYFZX2bQlmPGpzA4-OxPb5L1z72JsizSSwRsx1I8' # june 2026 goit update

# Pipeline type: 'Oil-NGL' | 'Oil' | 'NGL' | 'Gas' | 'Gas-Hydrogen' | 'Hydrogen' | 'Oil-and-Gas'
PIPELINE_TYPE = 'Oil-NGL'
#PIPELINE_TYPE = 'Oil'
#PIPELINE_TYPE = 'NGL'
#PIPELINE_TYPE = 'Oil-and-Gas'
#PIPELINE_TYPE = 'Gas'

# Fuel simplification: None | 'Oil' | 'NGL' | 'Oil-and-NGL' | 'Gas'
SIMPLIFY_FUELS = 'Oil-and-NGL'
#SIMPLIFY_FUELS = None
#SIMPLIFY_FUELS = 'Oil'
#SIMPLIFY_FUELS = 'NGL'
#SIMPLIFY_FUELS = 'Gas'

# Status filter (None = all statuses, or list of status values)
FILTER_STATUS = None
#FILTER_STATUS = ['operating']
#FILTER_STATUS = ['operating', 'construction']

# Geographic filter (None = all countries, or list of country names)
FILTER_COUNTRIES = None
#FILTER_COUNTRIES = ['Iran', 'Iraq', 'Saudi Arabia', 'Kuwait', 'United Arab Emirates', 'Qatar', 'Oman']

# Paths
# Routes come from a git worktree pinned to the 'normalized' branch of
# goit-ggit-pipeline-routes (the main checkout stays on whatever branch
# it's on). Refresh with: cd GOIT-GGIT-pipeline-routes-normalized && git pull
PIPELINE_ROUTES_PATH = (
    '/Users/baird/Dropbox/_git_ALL/_github-repos-gem/'
    'GOIT-GGIT-pipeline-routes-normalized/data/individual-routes/'
)
OUTPUT_DIR = 'data-files'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"✓ Config: {PIPELINE_TYPE}" + (f" | Status: {FILTER_STATUS}" if FILTER_STATUS else "") + (f" | Filter: {len(FILTER_COUNTRIES)} countries" if FILTER_COUNTRIES else ""))

## Helper Functions

In [5]:
def get_config(ptype):
    """Pipeline configuration lookup. Fuel buckets come from
    gem-tracker-constants; 'CO2' is kept inline because it isn't part
    of the canonical oil/NGL bucket but the GOIT export includes it."""
    configs = {
        'Oil-NGL': {
            'fuel_options': list(OIL_NGL_COMBINED) + ['CO2'],
            'folder': 'liquid-pipelines', 'tracker': 'GOIT',
            'sheet': 'Oil/NGL pipelines', 'dict_sheet': 'Data dictionary - Oil/NGL pipelines',
            'copyright_sheet': 'Copyright - GOIT'
        },
        'Oil': {
            'fuel_options': list(OIL_FUEL_OPTIONS),
            'folder': 'liquid-pipelines', 'tracker': 'GOIT',
            'sheet': 'Oil/NGL pipelines', 'dict_sheet': 'Data dictionary - Oil/NGL pipelines',
            'copyright_sheet': 'Copyright - GOIT'
        },
        'NGL': {
            'fuel_options': list(NGL_FUEL_OPTIONS),
            'folder': 'liquid-pipelines', 'tracker': 'GOIT',
            'sheet': 'Oil/NGL pipelines', 'dict_sheet': 'Data dictionary - Oil/NGL pipelines',
            'copyright_sheet': 'Copyright - GOIT'
        },
        'Gas': {
            'fuel_options': list(GAS_FUEL_OPTIONS),
            'folder': 'gas-pipelines', 'tracker': 'GGIT',
            'sheet': 'Gas pipelines', 'dict_sheet': 'Data dictionary - Gas pipelines',
            'copyright_sheet': 'Copyright - GGIT'
        },
        'Gas-Hydrogen': {
            'fuel_options': list(GAS_HYDROGEN_FUEL_OPTIONS),
            'folder': 'gas-pipelines', 'tracker': 'GGIT',
            'sheet': 'Gas pipelines', 'dict_sheet': 'Data dictionary - Gas pipelines',
            'copyright_sheet': 'Copyright - GGIT'
        },
        'Hydrogen': {
            'fuel_options': list(HYDROGEN_FUEL_OPTIONS),
            'folder': 'gas-pipelines', 'tracker': 'GGIT',
            'sheet': 'Gas pipelines', 'dict_sheet': 'Data dictionary - Gas pipelines',
            'copyright_sheet': 'Copyright - GGIT'
        },
        'Oil-and-Gas': {
            'fuel_options': list(OIL_NGL_COMBINED) + ['CO2'] + list(GAS_FUEL_OPTIONS),
            'folder': None, 'tracker': 'GOIT-GGIT', 'sheet': None, 'dict_sheet': None, 'copyright_sheet': None
        }
    }
    return configs[ptype]

def fetch_pipeline_data(ss, config):
    """Fetch and initial filter of pipeline data from a worksheet."""
    t0 = time.time()
    df = ss.worksheet('title', config['sheet']).get_as_df(start='A3', include_tailing_empty=False)
    df_dict = ss.worksheet('title', config['dict_sheet']).get_as_df(include_tailing_empty=False)
    df_copy = ss.worksheet('title', config['copyright_sheet']).get_as_df(include_tailing_empty=False)
    print(f"  Sheets fetched in {time.time() - t0:.1f}s")
    
    if df_copy.shape[1] > 1:
        df_copy = pd.DataFrame(df_copy.iloc[:, 0])
        
    # Standard initial filters
    mask = (
        df['Fuel'].isin(config['fuel_options']) &
        (df['Status'] != 'N/A') &
        (df['PipelineName'] != '') &
        (df['RouteAccuracy'] != '')
    )
    df = df[mask].copy()
    
    # Dataset-specific adjustments
    if config['tracker'] == 'GGIT':
        collapse_gas_and_hydrogen(df)
        
    return df, df_dict, df_copy

def filter_by_countries(df, countries, col='CountriesOrAreas'):
    """Filter dataframe by country list."""
    if not countries:
        return df
    pattern = '|'.join(re.escape(c) for c in countries)
    mask = df[col].astype(str).str.contains(pattern, na=False)
    return df[mask]

def filter_by_status(df, statuses, col='Status'):
    """Filter dataframe by status list."""
    if not statuses:
        return df
    mask = df[col].isin(statuses)
    filtered = df[mask]
    print(f"  Status filter {statuses}: {len(df)} → {len(filtered)} rows")
    return filtered

def simplify_fuel_types(df, strategy):
    """Relabel fuels using the canonical buckets from gem-tracker-constants
    and drop rows outside the requested category.

      - 'Oil':         rows with Fuel in OIL_FUEL_OPTIONS -> 'Oil'
      - 'NGL':         rows with Fuel in NGL_FUEL_OPTIONS -> 'NGL'
      - 'Oil-and-NGL': union of the two; rows in BOTH buckets
                       ('Oil, NGL', 'Oil, NGL, naphtha') are labeled
                       'Oil, NGL' so the dual-product signal survives
      - 'Gas':         rows with Fuel in GAS_FUEL_OPTIONS -> 'Gas'
    """
    if not strategy or strategy == 'None':
        return df

    oil_fuels, ngl_fuels = set(OIL_FUEL_OPTIONS), set(NGL_FUEL_OPTIONS)
    before = len(df)

    if strategy == 'Oil':
        df = df[df['Fuel'].isin(oil_fuels)].copy()
        df['Fuel'] = 'Oil'
    elif strategy == 'NGL':
        df = df[df['Fuel'].isin(ngl_fuels)].copy()
        df['Fuel'] = 'NGL'
    elif strategy == 'Oil-and-NGL':
        df = df[df['Fuel'].isin(oil_fuels | ngl_fuels)].copy()
        dual_mask = df['Fuel'].isin(oil_fuels & ngl_fuels)
        oil_mask = df['Fuel'].isin(oil_fuels) & ~dual_mask
        ngl_mask = df['Fuel'].isin(ngl_fuels) & ~dual_mask
        df.loc[dual_mask, 'Fuel'] = 'Oil, NGL'
        df.loc[oil_mask, 'Fuel'] = 'Oil'
        df.loc[ngl_mask, 'Fuel'] = 'NGL'
    elif strategy == 'Gas':
        df = df[df['Fuel'].isin(GAS_FUEL_OPTIONS)].copy()
        df['Fuel'] = 'Gas'

    dropped = before - len(df)
    if dropped:
        print(f"  Fuel simplification '{strategy}': dropped {dropped} rows with non-matching fuels")

    return df

def _count_geom_points(geom):
    """Count total coordinate points across all parts of a geometry."""
    if hasattr(geom, 'geoms'):
        return sum(_count_geom_points(g) for g in geom.geoms)
    coords = getattr(geom, 'coords', None)
    return len(coords) if coords is not None else 0

def _scan_route_files(routes_path):
    """Pre-scan a routes directory and return a set of available ProjectIDs."""
    try:
        return {os.path.splitext(f)[0] for f in os.listdir(routes_path) if f.endswith('.geojson')}
    except FileNotFoundError:
        return set()

def check_no_route_geojson_files(df, routes_path):
    """Check all 'no route' pipelines against geojson files on disk.
    
    Flags pipelines where the sheet says 'no route' but the geojson file
    in the GitHub routes directory has actual geometry. Separates into
    simple routes (<=5 points, should be nulled) and complex routes
    (>5 points, need review).
    """
    no_route_mask = df['RouteAccuracy'].str.lower().str.strip() == 'no route'
    no_route_pids = (
        df.loc[no_route_mask, 'ProjectID']
        .astype(str).str.strip()
        .loc[lambda s: (s != '') & s.notna()]
        .unique()
    )
    
    if len(no_route_pids) == 0:
        print("  No-route check: no 'no route' pipelines found in sheet")
        return
    
    available_files = _scan_route_files(routes_path)
    
    simple = []   # <=5 points
    complex_ = [] # >5 points
    read_errors = []
    for pid in no_route_pids:
        if pid not in available_files:
            continue
        route_file = routes_path / f"{pid}.geojson"
        try:
            gdf = gpd.read_file(route_file)
            if gdf.empty or gdf.geometry.isna().all():
                continue
            geom = gdf.geometry.union_all() if hasattr(gdf.geometry, 'union_all') else gdf.unary_union
            n_pts = _count_geom_points(geom)
            if n_pts <= 5:
                simple.append((pid, n_pts))
            else:
                complex_.append((pid, n_pts))
        except Exception as exc:
            read_errors.append((pid, str(exc)))
    
    if not simple and not complex_:
        print(f"  ✓ No-route check: all {len(no_route_pids)} 'no route' pipelines "
              f"have null/missing geojson files")
        if read_errors:
            print(f"  ⚠ {len(read_errors)} file(s) failed to read: {[pid for pid, _ in read_errors]}")
        return
    
    total = len(simple) + len(complex_)
    print(f"  ⚠ NO-ROUTE CONFLICT: {total} pipeline(s) marked 'no route' in sheet "
          f"but have geometry in geojson files:")
    
    if simple:
        print(f"\n    Simple routes (<=5 points, should be nulled) — {len(simple)}:")
        for pid, n in sorted(simple):
            print(f"      {pid} ({n} pts)")
    
    if complex_:
        print(f"\n    Complex routes (>5 points, need review) — {len(complex_)}:")
        for pid, n in sorted(complex_):
            print(f"      {pid} ({n} pts)")
    
    if read_errors:
        print(f"\n    ⚠ Failed to read {len(read_errors)} file(s):")
        for pid, reason in read_errors:
            print(f"      {pid}: {reason}")

def enforce_no_route_null_geometry(df):
    """Ensure pipelines with 'no route' RouteAccuracy have null geometry in export."""
    no_route_mask = df['RouteAccuracy'].str.lower().str.strip() == 'no route'
    no_route_count = no_route_mask.sum()
    if no_route_count == 0:
        return df
    
    df = df.copy()
    had_geometry = no_route_mask & df['geometry'].notna()
    bad_count = had_geometry.sum()
    
    if bad_count > 0:
        bad_ids = df.loc[had_geometry, 'ProjectID'].tolist()
        print(f"  ⚠ {bad_count} 'no route' pipeline(s) had non-null geometry — set to null: {bad_ids}")
    
    df.loc[no_route_mask, 'geometry'] = None
    print(f"  ✓ No-route export fix: {no_route_count} 'no route' pipelines set to null geometry")
    return df

def load_geometries(df, routes_path, name, progress_every=25):
    """Load route geometries for pipelines sequentially (GDAL is not thread-safe)."""
    df = df.copy()
    pid_series = df['ProjectID'].astype('string').str.strip()
    unique_ids = pid_series[pid_series.notna() & (pid_series != '')].unique().tolist()
    geom_map = {}
    missing = []
    failed = []
    start = time.time()
    
    # Pre-scan directory to avoid per-file existence checks
    available_files = _scan_route_files(routes_path)
    
    print(f"Loading {name} geometries ({len(unique_ids)} unique IDs, {len(available_files)} files on disk)...")
    
    for idx, pid in enumerate(unique_ids, 1):
        if pid not in available_files:
            missing.append(pid)
            continue
        
        route_file = routes_path / f"{pid}.geojson"
        try:
            gdf = gpd.read_file(route_file)
            if gdf.empty or gdf.geometry.isna().all():
                missing.append(pid)
                continue
            geom = gdf.geometry.union_all() if hasattr(gdf.geometry, 'union_all') else gdf.unary_union
            geom_map[pid] = set_precision(geom, grid_size=1e-6)
        except Exception as exc:
            failed.append((pid, str(exc)))
        
        if idx % progress_every == 0 or idx == len(unique_ids):
            elapsed = time.time() - start
            print(f"  {idx}/{len(unique_ids)} ({idx/len(unique_ids)*100:.0f}%) | elapsed {elapsed:.1f}s")
    
    df['geometry'] = pid_series.map(geom_map)
    print(f"  ✓ Loaded: {len(geom_map)}/{len(unique_ids)} unique files | Missing: {len(missing)} | Failed: {len(failed)}")
    if failed:
        sample = '; '.join([f"{pid}: {reason}" for pid, reason in failed[:10]])
        suffix = ' ...' if len(failed) > 10 else ''
        print(f"  Failed samples: {sample}{suffix}")
    
    df = enforce_no_route_null_geometry(df)
    return df

def export_files(gdf, base_name, dict_df=None, acronyms_df=None, copyright_df=None):
    """Export to Excel, GeoJSON, GeoPackage, and zipped Shapefile."""
    files = []
    t0 = time.time()
    
    # Excel
    xlsx = f"{base_name}.xlsx"
    with pd.ExcelWriter(xlsx, engine='openpyxl') as writer:
        gdf.drop(columns=['geometry']).to_excel(writer, sheet_name='Data', index=False)
        if dict_df is not None:
            dict_df.to_excel(writer, sheet_name='Data dictionary', index=False)
        if acronyms_df is not None:
            acronyms_df.to_excel(writer, sheet_name='Acronyms', index=False)
        if copyright_df is not None:
            copyright_df.to_excel(writer, sheet_name='Copyright', index=False)
    files.append(xlsx)
    
    # GeoJSON & GeoPackage (sequential — GDAL is not thread-safe)
    gjson = f"{base_name}.geojson"
    gpkg = f"{base_name}.gpkg"
    gdf.to_file(gjson, driver='GeoJSON')
    gdf.to_file(gpkg, driver='GPKG')
    files.extend([gjson, gpkg])
    
    # Shapefile, zipped (format limits: column names truncated to 10 chars,
    # text fields to 254 chars — the driver warns but writes everything)
    shp_dir = Path(f"{base_name}-shp")
    if shp_dir.exists():
        shutil.rmtree(shp_dir)
    shp_dir.mkdir()
    gdf.to_file(shp_dir / f"{Path(base_name).name}.shp", driver='ESRI Shapefile')
    shp_zip = f"{base_name}-shp.zip"
    with zipfile.ZipFile(shp_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for part in sorted(shp_dir.iterdir()):
            zf.write(part, part.name)
    shutil.rmtree(shp_dir)
    files.append(shp_zip)
    
    print(f"\n✓ Exported in {time.time() - t0:.1f}s: {', '.join([f.split('/')[-1] for f in files])}")
    return files

print("✓ Functions loaded")

✓ Functions loaded


---
## Part 1: Terminals

In [6]:
# # Load from Google Sheets
# gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
# ss = gc.open_by_key(TERMINALS_SHEET_KEY)

# terms_df = ss.worksheet('title', 'LNG export & import terminals').get_as_df(start='A3')
# terms_dict = ss.worksheet('title', 'Data dictionary - Terminals').get_as_df()
# terms_acro = ss.worksheet('title', 'Acronyms').get_as_df()
# terms_copy = ss.worksheet('title', 'Copyright - GGIT').get_as_df()
# if terms_copy.shape[1] > 1:
#     terms_copy = pd.DataFrame(terms_copy.iloc[:, 0])

# # Filter
# orig_count = len(terms_df)
# terms_df = terms_df[
#     (terms_df['TerminalType'] != 'Oil export terminal') &
#     (terms_df['TerminalName'] != '') &
#     (terms_df['Status'] != 'N/A')
# ]

# # Select columns for export
# cols_to_export = terms_dict[
#     (terms_dict['IncludeWithDataRelease'] == 'Yes') &
#     (terms_dict['DataReleaseColumnOrder'].notna())
# ].sort_values('DataReleaseColumnOrder')['VariableName'].tolist()

# terms_dict_export = terms_dict[
#     terms_dict['VariableName'].isin(cols_to_export)
# ][['VariableName', 'Definition']]

# # Create geometries
# terms_df['geometry'] = terms_df.apply(
#     lambda r: shapely.geometry.Point(r['Longitude'], r['Latitude']) 
#     if r['Longitude'] not in ['Unknown', 'TBD', ''] and r['Latitude'] not in ['Unknown', 'TBD', ''] 
#     else shapely.geometry.Point(),
#     axis=1
# )

# terms_gdf = gpd.GeoDataFrame(terms_df[cols_to_export], geometry=terms_df['geometry'], crs='EPSG:4326')

# print(f"✓ Terminals: {orig_count} → {len(terms_gdf)} (after filtering)")

# # Export (date stamp: year-month only)
# today = datetime.date.today()
# term_files = export_files(
#     terms_gdf,
#     f"{OUTPUT_DIR}/GEM-GGIT-LNG-Terminals-{today:%Y-%m}",
#     terms_dict_export, terms_acro, terms_copy
# )

---
## Part 2: Pipelines

In [7]:
t_total = time.time()

# Load based on mode
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
ss = gc.open_by_key(PIPELINES_SHEET_KEY)
config = get_config(PIPELINE_TYPE)

def process_single_dataset(ss, cfg, label):
    """Fetch, filter, and load geometries for one pipeline dataset."""
    df, df_dict, df_copy = fetch_pipeline_data(ss, cfg)
    raw_ids = set(df['ProjectID'].dropna().astype(str).str.strip()) - {''}
    
    print(f"\nChecking {label} no-route conflicts...")
    check_no_route_geojson_files(df, Path(PIPELINE_ROUTES_PATH) / cfg['folder'])
    
    df = filter_by_status(df, FILTER_STATUS)
    df = simplify_fuel_types(filter_by_countries(df, FILTER_COUNTRIES), SIMPLIFY_FUELS)
    df = load_geometries(df, Path(PIPELINE_ROUTES_PATH) / cfg['folder'], label)
    
    return df, df_dict, df_copy, raw_ids

if PIPELINE_TYPE == 'Oil-and-Gas':
    oil_cfg, gas_cfg = get_config('Oil-NGL'), get_config('Gas')
    
    oil_df, oil_dict, oil_copy, raw_oil_ids = process_single_dataset(ss, oil_cfg, 'Oil-NGL')
    gas_df, gas_dict, gas_copy, raw_gas_ids = process_single_dataset(ss, gas_cfg, 'Gas')
    
    raw_db_ids = raw_oil_ids | raw_gas_ids
    
    # Combine DataFrames
    oil_gdf = gpd.GeoDataFrame(oil_df, geometry='geometry', crs='EPSG:4326')
    gas_gdf = gpd.GeoDataFrame(gas_df, geometry='geometry', crs='EPSG:4326')
    common_cols = sorted(set(oil_gdf.columns) & set(gas_gdf.columns))
    pipes_gdf = pd.concat([oil_gdf[common_cols], gas_gdf[common_cols]], ignore_index=True)
    pipes_gdf = gpd.GeoDataFrame(pipes_gdf, geometry='geometry', crs='EPSG:4326')
    
    # Combine metadata
    common_fields = set(oil_dict['VariableName']) & set(gas_dict['VariableName'])
    pipes_dict = pd.concat([oil_dict, gas_dict[~gas_dict['VariableName'].isin(common_fields)]], ignore_index=True)
    pipes_copy = pd.DataFrame({'Copyright': ['=== GOIT ===', *oil_copy.iloc[:, 0].tolist(), '', '=== GGIT ===', *gas_copy.iloc[:, 0].tolist()]})
    pipes_acro = ss.worksheet('title', 'Acronyms').get_as_df(include_tailing_empty=False)

else:
    # Single dataset mode
    pipes_df, pipes_dict, pipes_copy, raw_db_ids = process_single_dataset(ss, config, PIPELINE_TYPE)
    pipes_gdf = gpd.GeoDataFrame(pipes_df, geometry='geometry', crs='EPSG:4326')
    pipes_acro = ss.worksheet('title', 'Acronyms').get_as_df(include_tailing_empty=False)

# Select and order columns for export
export_mask = (pipes_dict['IncludeWithDataRelease'] == 'Yes') & (pipes_dict['DataReleaseColumnOrder'].notna())
export_cols = pipes_dict[export_mask].sort_values('DataReleaseColumnOrder')['VariableName'].tolist()
export_cols = [c for c in export_cols if c in pipes_gdf.columns]

pipes_dict_export = pipes_dict[pipes_dict['VariableName'].isin(export_cols)][['VariableName', 'Definition']]
pipes_gdf_export = pipes_gdf[export_cols + ['geometry']].copy()

print(f"\n✓ Export ready: {len(pipes_gdf_export)} pipelines, {len(export_cols)} columns")
print(f"  Total cell time: {time.time() - t_total:.1f}s")

/Users/baird/miniconda3/envs/gem/lib/python3.13/site-packages/pygsheets/worksheet.py:1554: UserWarning: At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.
  warnings.warn('At least one column name in the data frame is an empty string. If this is a concern, please specify include_tailing_empty=False and/or ensure that each column containing data has a name.')


  Sheets fetched in 1.8s

Checking Oil-NGL no-route conflicts...
  ⚠ NO-ROUTE CONFLICT: 1 pipeline(s) marked 'no route' in sheet but have geometry in geojson files:

    Complex routes (>5 points, need review) — 1:
      P1955 (19 pts)
  Fuel simplification 'Oil-and-NGL': dropped 167 rows with non-matching fuels
Loading Oil-NGL geometries (1925 unique IDs, 2093 files on disk)...
  25/1925 (1%) | elapsed 0.0s
  50/1925 (3%) | elapsed 0.1s
  75/1925 (4%) | elapsed 0.2s
  100/1925 (5%) | elapsed 0.2s
  125/1925 (6%) | elapsed 0.3s
  150/1925 (8%) | elapsed 0.4s
  175/1925 (9%) | elapsed 0.5s
  200/1925 (10%) | elapsed 0.5s
  225/1925 (12%) | elapsed 0.6s
  250/1925 (13%) | elapsed 0.6s
  275/1925 (14%) | elapsed 1.2s
  300/1925 (16%) | elapsed 1.3s
  325/1925 (17%) | elapsed 1.3s
  350/1925 (18%) | elapsed 1.5s
  375/1925 (19%) | elapsed 1.6s
  400/1925 (21%) | elapsed 1.6s
  425/1925 (22%) | elapsed 1.6s
  450/1925 (23%) | elapsed 1.7s
  475/1925 (25%) | elapsed 1.7s
  500/1925 (26%) | e

### Compare database ProjectIDs with GitHub repo routes

In [8]:
# Map pipeline type to the relevant repo folder(s)
folder_map = {
    'Oil-NGL':      ['liquid-pipelines'],
    'Oil':          ['liquid-pipelines'],
    'NGL':          ['liquid-pipelines'],
    'Gas':          ['gas-pipelines'],
    'Gas-Hydrogen': ['gas-pipelines', 'hydrogen-pipelines'],
    'Hydrogen':     ['hydrogen-pipelines'],
    'Oil-and-Gas':  ['liquid-pipelines', 'gas-pipelines', 'hydrogen-pipelines'],
}

def normalize_ids(series):
    return set(series.dropna().astype(str).str.strip()) - {''}

routes_base = Path(PIPELINE_ROUTES_PATH)
db_ids = normalize_ids(pipes_gdf['ProjectID'])

# Collect repo ProjectIDs from the relevant folder(s)
repo_ids = set()
for folder in folder_map[PIPELINE_TYPE]:
    folder_path = routes_base / folder
    try:
        files = os.listdir(folder_path)
        folder_ids = {os.path.splitext(f)[0].strip() for f in files if f.endswith('.geojson')}
        print(f"{folder}: {len(folder_ids)} .geojson files (exists: {folder_path.exists()})")
    except Exception as e:
        folder_ids = set()
        print(f"{folder}: ERROR - {e}")
    repo_ids |= folder_ids

# raw_db_ids was cached in the previous cell (avoids re-fetching Google Sheets)

# Compare
in_db_not_repo = sorted(db_ids - repo_ids)
in_repo_not_db = sorted(repo_ids - db_ids)
in_repo_raw_db = sorted(pid for pid in in_repo_not_db if pid in raw_db_ids)
in_repo_missing_raw = sorted(pid for pid in in_repo_not_db if pid not in raw_db_ids)

print(f"\nDatabase (filtered): {len(db_ids)} | Raw sheet: {len(raw_db_ids)} | Repo: {len(repo_ids)} | In both: {len(db_ids & repo_ids)}")

print()
if in_db_not_repo:
    print(f"In DATABASE but NOT in repo ({len(in_db_not_repo)}):")
    for pid in in_db_not_repo:
        print(f"  {pid}")
else:
    print("All database ProjectIDs have a .geojson in the repo.")

print()
if in_repo_not_db:
    print(f"In REPO but NOT in database ({len(in_repo_not_db)}):")
    for pid in in_repo_not_db:
        print(f"  {pid}")
    print()
    if in_repo_raw_db:
        print(f"  Present in raw Google Sheet before filtering ({len(in_repo_raw_db)}):")
        for pid in in_repo_raw_db:
            print(f"    {pid}")
    else:
        print("  None are present in the raw Google Sheet before filtering.")
    print()
    if in_repo_missing_raw:
        print(f"  Missing entirely from the raw Google Sheet ({len(in_repo_missing_raw)}):")
        for pid in in_repo_missing_raw:
            print(f"    {pid}")
    else:
        print("  None are missing entirely from the raw Google Sheet.")
else:
    print("All repo .geojson files have a matching ProjectID in the database.")

liquid-pipelines: 2093 .geojson files (exists: True)

Database (filtered): 1925 | Raw sheet: 2092 | Repo: 2093 | In both: 1922

In DATABASE but NOT in repo (3):
  P7994
  P7995
  P7996

In REPO but NOT in database (171):
  P0006
  P0108
  P0115
  P0136
  P0633
  P0825
  P0826
  P1093
  P1212
  P1217
  P1441
  P1443
  P1444
  P1445
  P2200
  P2220
  P2223
  P2226
  P2483
  P2647
  P2739
  P3700
  P3717
  P3722
  P3846
  P3866
  P3880
  P3890
  P5130
  P5147
  P5239
  P5360
  P5361
  P6045
  P6068
  P6069
  P6082
  P6102
  P6132
  P6148
  P6149
  P6182
  P6183
  P6295
  P6308
  P6309
  P6350
  P6374
  P6375
  P6378
  P6379
  P6401
  P6409
  P6439
  P6445
  P6452
  P6482
  P6483
  P6492
  P6549
  P7154
  P7157
  P7158
  P7159
  P7184
  P7185
  P7186
  P7188
  P7190
  P7191
  P7201
  P7202
  P7203
  P7219
  P7220
  P7222
  P7244
  P7246
  P7248
  P7249
  P7250
  P7251
  P7252
  P7253
  P7254
  P7255
  P7257
  P7258
  P7262
  P7270
  P7271
  P7272
  P7273
  P7274
  P7275
  P7276
  P7284
  P

In [ ]:
# Export pipelines
today = datetime.date.today()
base_name = f"{OUTPUT_DIR}/GEM-{config['tracker']}-{PIPELINE_TYPE}-Pipelines"

# Add status suffix if filtering
if FILTER_STATUS:
    status_suffix = '-'.join([s.replace(' ', '') for s in FILTER_STATUS])
    base_name += f"-{status_suffix}"

# Add country suffix if filtering
if FILTER_COUNTRIES:
    country_suffix = '-'.join([c.replace(' ', '') for c in FILTER_COUNTRIES])
    base_name += f"-{country_suffix}"

# Date stamp: year-month only
base_name += f"-{today:%Y-%m}"

pipe_files = export_files(
    pipes_gdf_export,
    base_name,
    pipes_dict_export, pipes_acro, pipes_copy
)

---
## Summary

In [ ]:
print("=" * 60)
print("EXPORT COMPLETE")
print("=" * 60)
print(f"Date: {today}")
#print(f"\nTerminals: {len(terms_gdf)}")
print(f"Pipelines: {len(pipes_gdf_export)} ({PIPELINE_TYPE})" + (f" | Simplified: {SIMPLIFY_FUELS}" if SIMPLIFY_FUELS else ""))
if FILTER_STATUS:
    print(f"Status: {', '.join(FILTER_STATUS)}")
if FILTER_COUNTRIES:
    print(f"Countries: {', '.join(FILTER_COUNTRIES)}")
#print(f"\nFiles: {len(term_files) + len(pipe_files)}")
print(f"\nFiles: {len(pipe_files)}")
print("=" * 60)

EXPORT COMPLETE
Date: 2026-06-11
Pipelines: 1938 (Oil-NGL) | Simplified: Oil-and-NGL

Files: 3
